In [ ]:
import os, cv2
import numpy as np

video_path = "Dataset\\cam1\\1.avi"
# video_path = "1_.avi"
output_path = video_path+"_color_tracked.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"Не удалось открыть видео: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30

# Настройка колонок для формата 1624х1240 46к/с (ДАТАСЕТ ТАКОЙ :)
row1_width = 200
row2_width = 420
row3_width = 480
row4_width = 420
row5_width = frame_width - (row1_width + row2_width + row3_width + row4_width)

x2 = row1_width
x3 = x2 + row2_width
x4 = x3 + row3_width
x5 = x4 + row4_width

# Цветовой фильтр. цыпленок детектится при появлении сразу
lower_yellow = np.array([10, 20, 110])   # Оттенок желтый H 10-40, насыщенность S 20-200
upper_yellow = np.array([40, 200, 255])  # Яркость V 60-255. 60 на темном датасете
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

# Инициализация
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (812, 620))
seen_ids = {"row2": set(), "row3": set(), "row4": set()}
next_id = 0
tracks = {}  # {id: [cx, cy, frame, zone]}

frame_count = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # frame = cv2.flip(frame, -1)  # flipCode = -1 → поворот на 180°
    frame[0:60, :] = 0  # Закрашиваем верхнюю часть кадра (60 пикселей) чёрным цветом (от ложных срабатываний отжелезки сверху).
    # frame = cv2.medianBlur(frame,25)  # медианное размытие. Не помогло.
    annotated_frame = frame.copy()
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    current_objects = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 30000:  # площадь объекта
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        cx = x + w // 2
        cy = y + h // 2

        if x2 <= cx < x3:
            zone = "row2"
        elif x3 <= cx < x4:
            zone = "row3"
        elif x4 <= cx < x5:
            zone = "row4"
        else:
            continue

        current_objects.append((cx, cy, w, h, zone))
        color = (255, 0, 0) if zone == "row2" else (0, 255, 0) if zone == "row3" else (0, 0, 255)
        cv2.rectangle(annotated_frame, (x, y), (x+w, y+h), color, 4)

    # Трекинг с привязкой к колонке
    new_tracks = {}
    for cx, cy, w, h, zone in current_objects:
        matched = False
        for tid, (prev_cx, prev_cy, last_frame, prev_zone) in tracks.items():
            if zone == prev_zone and prev_cy < cy:
                new_tracks[tid] = (cx, cy, frame_count, zone)
                matched = True
                break

        if not matched:
            seen_ids[zone].add(next_id)
            new_tracks[next_id] = (cx, cy, frame_count, zone)
            next_id += 1

    tracks = new_tracks

    # Отрисовка
    cv2.line(annotated_frame, (x2, 0), (x2, frame_height), (255, 255, 255), 1)
    cv2.line(annotated_frame, (x3, 0), (x3, frame_height), (255, 255, 255), 1)
    cv2.line(annotated_frame, (x4, 0), (x4, frame_height), (255, 255, 255), 1)
    cv2.line(annotated_frame, (x5, 0), (x5, frame_height), (255, 255, 255), 1)

    for tid, (cx, cy, last_frame, zone) in tracks.items():
        color = (255, 0, 0) if zone == "row2" else (0, 255, 0) if zone == "row3" else (0, 0, 255)
        cv2.putText(annotated_frame, f"ID:{tid}", (int(cx), int(cy)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)

    # УМЕНЬШАЕМ КАДР ПОСЛЕ ОБРАБОТКИ ДЛЯ ВЫХОДНОГО ВИДЕО
    annotated_frame = cv2.resize(annotated_frame, (812, 620), interpolation=cv2.INTER_AREA)
    out.write(annotated_frame)
    frame_count += 1
    if frame_count % 440 == 0:  # печатаем примерно раз в 10 секунд (440 фреймов)
        print(f"Кадр {frame_count}: объектов = {len(current_objects)}")

cap.release()
out.release()

# Вывод
row2_count = len(seen_ids["row2"])
row3_count = len(seen_ids["row3"])
row4_count = len(seen_ids["row4"])
total = row2_count + row3_count + row4_count
print("\n" + "="*60)
print(f"✅ Колонка 2 (левый): {row2_count}")
print(f"✅ Колонка 3 (центр): {row3_count}")
print(f"✅ Колонка 4 (правый): {row4_count}")
print(f"✅ ВСЕГО: {total}")
print("="*60)
# добавляем в имя файла total
os.rename(output_path, f"{'.'.join(output_path.split('.')[:-1])}_{total}.{output_path.split('.')[-1]}")  

Кадр 440: объектов = 0
Кадр 880: объектов = 0
Кадр 1320: объектов = 0
Кадр 1760: объектов = 0
Кадр 2200: объектов = 0
Кадр 2640: объектов = 0
Кадр 3080: объектов = 0
Кадр 3520: объектов = 0
Кадр 3960: объектов = 0
Кадр 4400: объектов = 0
Кадр 4840: объектов = 0
Кадр 5280: объектов = 1

✅ Колонка 2 (левый): 88
✅ Колонка 3 (центр): 119
✅ Колонка 4 (правый): 62
✅ ВСЕГО: 269
